In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 9. 데이터 그룹화 및 집계

In [ ]:
from statsmodels.api import datasets
iris = datasets.get_rdataset('iris').data
iris

## 9.1 Group by
 - cf. pivot_table
 
 - df. groupby('label1')['label2'].func()
     - label1 : 기준이 된 열 이름
     - label2 : 생략 시 모든 열에 대해 적용
     - func() : 기초통계함수
 - 그룹화의 기준이 된 열 이름(column)은 그룹화 이후 행 이름(Index_Name)이 됨

In [ ]:
# 'Species'별로 데이터를 그룹화한 뒤, 각 그룹의 합계 구하기
iris.groupby('Species').sum()

In [ ]:
# 1) 그룹화 방법 1
# df.groupby('label1')['label2'].func() : laber1별로 그룹화한 뒤, laber2의 데이터 출력
# df.groupby('label1')[['label2','laber3']].func() : laber1별로 그룹화한 뒤, laber2, laber3의 데이터 출력
iris_grouped = iris.groupby('Species')
iris_grouped_sum = iris_grouped.sum()
iris_grouped_sum

In [ ]:
#iris_grouped_sum의 부분 조회 1
iris_grouped_sum.loc[['setosa', 'virginica']]

In [ ]:
# 그룹화의 기준이 된 열 이름(column)은 그룹화 이후 행 이름(Index_Name)이 됨
print(iris_grouped_sum.index)

In [ ]:
iris_grouped_sum = iris_grouped_sum.reset_index()

In [ ]:
iris_grouped_sum

In [ ]:
#iris_grouped_sum의 부분 조회 2
iris_grouped_sum[(iris_grouped_sum.Species=='setosa')|(iris_grouped_sum.Species=='virginica')]

In [ ]:
#iris_grouped_sum의 부분 조회 3
iris_grouped_sum[iris_grouped_sum.Species.isin(['setosa', 'virginica'])]

In [ ]:
# 1) 그룹화 방법 1-1 : 그룹핑 과정에서 index를 컬럼 데이터로 편입하기
# df.groupby(by='label1', as_index=False)['label2'].func()

iris.groupby('Species', as_index=False).sum()

In [ ]:
# 종별 Sepal.Length와 Sepal.Width의 평균
iris.groupby('Species').mean()[['Sepal.Length','Sepal.Width']]
iris.groupby('Species')[['Sepal.Length','Sepal.Width']].mean()

In [ ]:
# 종별 Sepal.Length의 합계
iris.groupby('Species')[['Sepal.Length']].sum()

- df.pivot_table(index='labels1',values='labels2',aggfunc='func')
    - label1 : 기준이 된 열 이름
    - label2 : 함수를 적용해 출력할 열 (생략 시 모든 열에 대해 적용)
    - aggfunc : 적용할 함수 (기본값 : mean)
    
* pivot_table - 오름차순으로 자동 정렬
* groupby() - 원본 열 순서 유지

In [ ]:
# 그룹화 방법 2
# df.pivot_table(index='labels1',values='labels2',aggfunc='func')

g = iris.pivot_table(index='Species')
g

In [ ]:
# 범례를 바깥으로 빼기
import matplotlib.pyplot as plt
g.plot(kind='bar', figsize=(10,4), rot=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
# pivot_talbe과 열 순서가 다름
g2 = iris.groupby('Species').mean()
g2

In [ ]:
import matplotlib.pyplot as plt
g2.plot(kind='bar', figsize=(10,4), rot=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
iris.pivot_table(index='Species', # Species 별로 나눈 뒤
                values='Sepal.Length', # Sepal.Length 값들에게서
                aggfunc='count') # count 값을 반환

In [ ]:
iris['Species'].value_counts() #count 값을 정렬하여 반환

In [ ]:
# 임의의 각 열에 결측치 할당(0열 - 결측치 1 / 1열 - 결측치 2 / 2열 - 결측치 3 /...)
import numpy as np
import random


for col in range(iris.shape[1]-1): # iris의 열 수 - 1
    rows = random.sample(range(len(iris)), col+1)
    iris.iloc[rows, col] = np.nan

In [ ]:
# 열별 결측치를 제외한 데이터 개수
iris.groupby('Species').count()

### 다중열 그룹화

In [ ]:
#iris로 다중열 그룹화하기 위해 num 컬럼 추가 (150행 중, Species 3그룹, num 6그룹)

numbers = np.array([[i]*25 for i in range(6)]).flatten()

In [ ]:
iris['num'] = numbers

In [ ]:
# num 열 추가 확인
iris.head(1)

In [ ]:
# 2 level row index
r = iris.groupby(['Species','num']).sum()
r

In [ ]:
# setosad와 virginica의 합계
r.loc[['setosa','virginica']]

In [ ]:
# setosad, 0과 virginica, 5의 합계
r.loc[[('setosa',0),('virginica',5)]]

In [ ]:
# 2 level row index
iris.pivot_table(index=['Species','num'])

In [ ]:
# 종별 요약통계량
iris.groupby(['Species']).describe().T

## 9.2 그룹간 데이터 처리

In [ ]:
from statsmodels.api import datasets
iris = datasets.get_rdataset('iris').data
iris.head(1)

In [ ]:
iris_grouped = iris.groupby('Species')
list(iris_grouped) # 나열 가능한 자료 여부 확인

In [ ]:
for speciesname, group in iris_grouped:
    print(speciesname)
    display(group.sample(3).sort_index().reset_index(drop=True))

In [ ]:
for speciesname, group in iris_grouped:
    print(speciesname)
    display(group.iloc[:45:15])

## 9.3 그룹별 인덱싱

In [ ]:
iris.groupby('Species').take([0,15,30,45])

# 10. 데이터 구조 변경
- 와이드 포맷 ↔ 롱 포맷
    - 와이드 포맷 : 가로로 긴 형식의 데이터 구조
    - 롱 포맷 : 세로로 긴 형식의 데이터 구조
    
- melt()

In [ ]:
from statsmodels.api import datasets
airquality = datasets.get_rdataset('airquality').data #1973년 뉴욕 공기질 데이터셋
airquality

## 10.1 melt() 언피벗팅
- 와이드 포맷 → 롱 포맷
- df.melt(id_vals, value_vars, var_name, value_name)
- pd.melt(df, id_vars, value_vars, var_name, value_name)

In [ ]:
airquality_melted =  airquality.melt(id_vars=['Month', 'Day'],
                                    #value_vars=['Ozone', 'Solar.R', 'Wind', 'Temp']
                                    )
airquality_melted

In [ ]:
airquality_melted =  airquality.melt(id_vars=['Month', 'Day'],
                                    var_name='변수명', # melt된 데이터의 열 이름
                                    value_name='값'
                                    )
airquality_melted = airquality_melted.sort_values(by=['Month', 'Day']).reset_index(drop=True)

## 10.2 pivot_table() 피벗팅
- 롱 포맷 → 와이드 포맷
- df.pivot_table(index, columns, values)
- pd.pivot_table(df, index, columns, values)

In [ ]:
airquality2 = airquality_melted.pivot_table(index=['Month', 'Day'],
                             columns=['변수명'],
                             #values='값'
                                           )
airquality2.columns = airquality2.columns.droplevel(level=0) # columns의 level 조정하기
airquality2.columns.name = None # columns의 name 제거
airquality2 = airquality2.reset_index()
airquality2

# 11. 데이터 프레임 함수
- apply(func, axis)
    - 시리즈: 요소별 func 적용
    - 데이터프레임: 행(axis=1) 또는 열(axis=0) 단위 func 적용
- map(func) : 각 요소별 func 적용 (시리즈만 적용 가능)
- applymap(func) : 각 요소별 func 적용 (데이터 프레임만 적용 가능)

## 11.1 apply() / map() 

In [ ]:
import pandas as pd
member = pd.read_csv('data/ch11_member.csv', comment='#')
member

In [ ]:
get_year = lambda datestr : int(datestr.split('-')[0])
get_month = lambda datestr : int(datestr.split('-')[1])

In [ ]:
# 시리즈에서의 apply / map 사용
member['Year'] = member['Birth'].map(lambda datestr : int(datestr.split('-')[0]))
member['Month'] = member['Birth'].apply(lambda datestr : int(datestr.split('-')[1]))

In [ ]:
member

In [ ]:
%ls "C:/ai/downloads/shareData/분양가격"

In [ ]:
df_last = pd.read_csv('C:/ai/downloads/shareData/분양가격/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260531 (1).csv', encoding = 'cp949')
df_last.head()

In [ ]:
def year_month(series):
    return "{}-{:02}".format(series['연도'], series['월'])
#    return str(series['연도'])+"-"+(str(series['월']) if series['월']>=10 else ("0"+str(series['월'])))
year_month(df_last.loc[300])

In [ ]:
# 데이터 프레임에서의 apply(func, axis) 사용
df_last['연월'] = df_last.apply(year_month, axis = 1)
df_last.tail()

In [ ]:
import numpy as np
X = iris.iloc[:,:-1]
avg = np.mean(X, axis = 0)
avg = X.mean()
avg

In [ ]:
# 각 열별 평균과의 차이를 출력
X.apply(lambda x : x - avg, axis = 1)

## 11.2 applymap()

In [ ]:
X.applymap(lambda x : round(x, 2))

### 함수 적용시키기 연습 예제
* 2013~2015
    * 와이드포맷 → 롱포맷
    * 평당 분양가격 / 3.3  → 분양가격 (제곱미터당 천원)
    * 2013년12월 → 연도 : 2013, 월 : 12
    * 지역 → 지역명
* 2015~
    * 규모구분 '모든면적' 데이터만 추출
* 2013~
    * 두 자료 합치기

In [ ]:
df_last = pd.read_csv('C:/ai/downloads/shareData/분양가격/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260531 (1).csv', encoding = 'cp949')
df_last.head()

In [ ]:
df_first = pd.read_csv('c:/ai/downloads/shareData/분양가격/전국 평균 평당 분양가격(2013년 9월부터 2015년 8월까지).csv', encoding='cp949')
df_first.head()

In [ ]:
df_first_melted = df_first.melt(id_vars='지역',
                                var_name='연월',
                                value_name='분양가격')
df_first_melted.head()

In [ ]:
df_first_melted['연도'] = df_first_melted['연월'].map(lambda datestr : int(datestr[:4]))
df_first_melted['월'] = df_first_melted['연월'].map(lambda datestr : int(datestr[5:-1]))

In [ ]:
df_first_melted['분양가격(제곱미터당 천원)'] = df_first_melted['분양가격'].map(lambda x : int(round( x / 3.3)))

In [ ]:
df_first_melted = df_first_melted.drop('분양가격', axis=1)

In [ ]:
df_first_melted = df_first_melted.drop('연월', axis=1)
df_first_melted['지역명'] = df_first_melted['지역']
df_first_melted = df_first_melted.drop('지역', axis=1)
# df_first_melted.rename(columns={'지역':'지역명'}, inplace = True) 컬럼명 변경하기
new_first = df_first_melted

In [ ]:
new_first.sort_index(axis = 'columns', ascending = False).head()

In [ ]:
df_last_finded = df_last.loc[df_last['규모구분'].str.find('모든면적')!=-1]
df_last_finded = df_last_finded.drop('규모구분', axis=1)

In [ ]:
new_last = df_last_finded

In [ ]:
new_last.sort_index(axis = 'columns', ascending = False).head()

In [ ]:
new_list = pd.concat([new_first, new_last]).sort_values(by=['연도', '월']).reset_index(drop=True)

In [ ]:
new_list.loc[::200]

# 12. 일괄 변경하기

* np.isnan(scalar) : nan 값을 결측치로 인식, 결측치 여부 확인 (값이 숫자형이어야 함)
* pd.isna(scalar), pd.isnull(scalar) : nan과 None을 보두 결측치로 인식, 결측치 여부 확인

## 12.1 fillna 
* 결측치 → 특정 값
* 일반적으로 결측치는 Drop 또는 평균값, 중위수, 최빈값, 그룹별 평균값, 이전/다음값으로 대체함

In [ ]:
df = pd.DataFrame([[np.nan, 2, np.nan, 0],
                   [np.nan, 5, np.nan, 5],
                   [np.nan, 2, np.nan, 4],
                   [1, np.nan, np.nan, 2]], columns=list('ABCD'))
df

In [ ]:
# 결측치를 0으로 대체
df.fillna(0)

In [ ]:
# 결측치를 이전 행의 값으로 대체
df.fillna(method='ffill')

In [ ]:
# 결측치를 다음 행의 값으로 대체
df.fillna(method='bfill')

In [ ]:
# 결측치를 열마다 특정 값으로 대체
values = {'A':99,'B':85,'C':75,'D':35}
df.fillna(value=values)

In [ ]:
# 결측치를 각 열의 평균값으로 대체 (대체 개수 제한 : 2개)
df.fillna(value=round(df.mean(axis=0)), limit=2)

In [ ]:
# 결측치를 각 열의 중위값으로 대체
df.fillna(value=df.median(axis=0))

In [ ]:
# 결측치를 특정 열의 최빈값으로 대체
df.mode() # 최빈값을 0행에 출력
df.fillna(value=df.mode().iloc[0])

## 12.2 replace
* replace(to_value, new_value, inplace=T/F)
* to_value를 new_value로 변경
    * to_value : 리스트, 딕셔너리, 정규표현식 적용 가능
    * new_value : 리스트 적용 가능

In [ ]:
s = pd.Series([0,1,2,3,None])
s.replace(np.nan, 0, inplace=True)
s

In [ ]:
s.replace([0,1], 9)

In [ ]:
# to_value의 리스트(인덱스)에 요소와 겹치는 값이 있을 경우, 함께 변경됨 (idx[4] : 0.0 → 6.0)
s.replace([0,1,2],[6,7,8])

In [ ]:
df = pd.DataFrame({'A':['bat','foo','bait'],
                  'B':['abc','bar','xyz']})
df

In [ ]:
# to_value에 정규표현식이 오는 경우
df.replace('ba.+', 'XX', regex = True)
df.replace('^ba.+$', 'XX', regex = True)
df.replace(regex='^ba.+$', value='XX')

In [ ]:
df.replace({'A':'bat','B':'abc'}, 'XX')

## 12.3 where/mask

* where(조건)

* mask(조건)

In [ ]:
# numpy where
a = np.array([1,2,3,4,5,6])
np.where(a>3)

In [ ]:
s = pd.Series(range(10,20))
s

In [ ]:
# 시리즈에서, 15 이상의 숫자만 반환
s[s>=15]

In [ ]:
# 시리즈에서, 조건이 만족되지 않은 데이터는 결측치 처리
s.where(s>=15)

In [ ]:
# 시리즈에서, 조건이 만족되지 않은 데이터는 특정 값으로 대체
s.where(s>=15, other=-1)

In [ ]:
# 시리즈에서, 조건을 만족하는 데이터는 결측치 처리
s.mask(s<15)

## 12.4 dropna
- 결측치가 있는 데이터 누락 (행/열 기준)

In [ ]:
df = pd.DataFrame({'name':['홍','김',np.nan,np.nan],
                  'age':[np.nan,20,30,np.nan],
                  'born':['서울','부산',np.nan,np.nan]})
df

In [ ]:
# 결측치가 존재하는 열(axis=1) Drop
df.dropna(axis=1) # 기본값 axis=0

In [ ]:
# 결측치가 존재하는 행 Drop
df.dropna()

In [ ]:
# 결측치가 아닌 값이 2개 이상일 경우 유지
df.dropna(thresh=2)

In [ ]:
# 'name', 'born' 열에 결측치가 존재하는 열 Drop
df.dropna(subset=['name', 'born'])

In [ ]:
# 모든 요소가 결측치인 행 Drop
df.dropna(how='all')

### iris 결측치를 종별 평균으로 대체

In [ ]:
import seaborn as sns
iris = sns.load_dataset('iris')
iris.head()

In [ ]:
# 각 열마다 20개 씩 결측치 할당
import random
for n in range(iris.shape[1]-1):
    iris.iloc[random.sample(range(iris.shape[0]),20), n] = np.nan

In [ ]:
# 각 열의 결측치 개수
iris.isna().sum()

In [ ]:
# 각 열의 결측치가 아닌 데이터 갯수
iris.notna().sum()

In [ ]:
iris[iris['sepal_length'].isna()|iris['sepal_width'].isna()|iris['petal_width'].isna()|iris['petal_length'].isna()]

In [ ]:
# 결측치가 있는 열 출력
iris[iris.isna().any(axis=1)]

In [ ]:
# 결측치를 열 평균으로 대체
iris.iloc[:,:-1].mean(axis=0)

In [ ]:
iris_fillna = iris.fillna(value=iris.iloc[:,:-1].mean(axis=0))
iris_fillna.isna().sum()

In [ ]:
iris_fillna.iloc[:,:-1] = iris_fillna.iloc[:,:-1].applymap(lambda x : np.round(x, 3))

In [ ]:
iris_fillna

In [ ]:
iris.isna().sum()

In [ ]:
iris_grouped = iris.groupby('species')

In [ ]:
iris_mean = iris_grouped.mean()

In [ ]:
iris_mean.loc['setosa','sepal_length']

In [ ]:
def fillna_grouped_mean(row):
    row = row.copy()
    for col, item in row.items():
        # item이 결측치일 경우, iris_mean.loc[종이름, col]로 대체
        if pd.isna(item):
            species = row['species']
            row[col] = iris_mean.loc[species, col].round(1) # 종+열별 평균값 
    return row
            
fillna_grouped_mean(iris.iloc[0])

In [ ]:
iris.apply(fillna_grouped_mean, axis=1)

# 13. Series
* 1차원 데이터

In [ ]:
s = pd.Series([1000,2000,3000], index=['banana','mango','apple'])
s

In [ ]:
# 요소 찾기
s['banana']

In [ ]:
# 요소 추가
s['watermelon']=20000
s

In [ ]:
# 요소 제거
s.drop('watermelon', inplace=True)
s

In [ ]:
# 인덱스를 기준으로 한 정렬
s = s.sort_index()
s

In [ ]:
# 값을 기준으로 한 정렬
s.sort_values()

In [ ]:
# 열 이름 지정 및 데이터 프레임으로 전환
s.to_frame('price')

In [ ]:
# 열 이름 지정 및 데이터 프레임으로 전환
s.to_frame('price').T

# 연습문제

## 실습형

In [ ]:
import seaborn as sns
iris = sns.load_dataset('iris')

In [ ]:
#1
iris.head()

In [ ]:
#2-1
X = iris.iloc[:,:-1]

In [ ]:
#2-2
y = iris.iloc[:,-1::]

In [ ]:
#3
temp = iris.iloc[0:50]

In [ ]:
#4 describe?
temp.info()

In [ ]:
#5
iris_versicolor = iris[iris['species']=='versicolor']

In [ ]:
#6
iris_df = pd.concat([X, y], axis=1)

In [ ]:
#7
iris.mean(numeric_only=True)

In [ ]:
#8
iris.corr(numeric_only=True)

In [ ]:
#9
avg = iris.mean(numeric_only=True)
X.apply(lambda x : x - avg, axis = 1).head()

In [ ]:
#10
iris_grouped = iris.groupby('species')
for speciesname, group in iris_grouped:
    print(speciesname)
    print(group.mean(numeric_only=True))
    print()

In [ ]:
iris_mean.loc['setosa']

In [ ]:
#11
iris_mean = iris_grouped.mean()

def ex_mean(row):
    row = row.copy()
    for col, item in row.items():
        species = row['species']
        if col == 'species':
            row[col] = np.nan
        else:
            row[col] = np.round(item - iris_mean.loc[species, col], 3)
    return row

In [ ]:
#11-1 예제와 열이름 순서 다름
for idx in range(0,len(iris),50):
    display(iris.apply(ex_mean, axis=1).iloc[idx:idx+3])

In [ ]:
#11-2 예제와 열이름 순서 다름
ex_mean1 = iris.apply(ex_mean, axis=1)[0:3]
ex_mean2 = iris.apply(ex_mean, axis=1)[50:53]
ex_mean3 = iris.apply(ex_mean, axis=1)[100:103]
pd.concat([ex_mean1, ex_mean2, ex_mean3])

In [ ]:
#11-3 강사님 피드백
iris_mean = iris.apply(lambda row : row - iris_mean.loc[row.species], axis=1)

In [ ]:
#11-3 강사님 피드백
ex_mean1 = iris_mean[0:3]
ex_mean2 = iris_mean[50:53]
ex_mean3 = iris_mean[100:103]
pd.concat([ex_mean1, ex_mean2, ex_mean3])